# Amazon Electronics Dataset — Exploratory Data Analysis

This notebook explores the Amazon Product Reviews 2023 Electronics subset.
Findings here directly inform modeling choices in later phases.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime

from src.data.loader import load_interactions, load_items, build_dataset

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

In [ ]:
# Load (will download + process on first run, ~5 min)
interactions, items = build_dataset()
print(f'Interactions: {interactions.shape}')
print(f'Items:        {items.shape}')
interactions.head()

## 1. Interaction Matrix Sparsity

The user-item matrix is almost always extremely sparse in recommendation systems.
High sparsity (>99%) confirms collaborative filtering needs implicit feedback signals,
not explicit rating-based methods.

In [ ]:
n_users = interactions['user_id'].nunique()
n_items = interactions['item_id'].nunique()
n_interactions = len(interactions)
max_possible = n_users * n_items
sparsity = 1 - (n_interactions / max_possible)

print(f'Unique users:       {n_users:,}')
print(f'Unique items:       {n_items:,}')
print(f'Interactions:       {n_interactions:,}')
print(f'Max possible:       {max_possible:,}')
print(f'Sparsity:           {sparsity:.4%}')

## 2. Rating Distribution

Amazon reviews are heavily skewed toward 5-star ratings (J-shaped distribution).
This means raw ratings are a poor signal — **implicit feedback** (did the user interact at all?)
is more reliable. We use ALS with implicit feedback accordingly.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
interactions['rating'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Rating')
ax.set_ylabel('Count')
ax.set_title('Rating Distribution')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
plt.tight_layout()
plt.show()

## 3. User Activity Distribution (Log Scale)

User activity follows a long-tail distribution: the majority of users have just 5–10
interactions (our minimum threshold), while a small number are power users with 100+.
This motivates the TTL-based Redis session store — most users are infrequent.

In [ ]:
user_counts = interactions.groupby('user_id').size()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(user_counts, bins=50, log=True, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of interactions per user')
ax.set_ylabel('Number of users (log scale)')
ax.set_title('User Activity Distribution')
plt.tight_layout()
plt.show()

print(f'Median interactions/user: {user_counts.median():.0f}')
print(f'Mean   interactions/user: {user_counts.mean():.1f}')
print(f'Max    interactions/user: {user_counts.max()}')
print(f'Users with 50+  interactions: {(user_counts >= 50).sum():,}')
print(f'Users with 100+ interactions: {(user_counts >= 100).sum():,}')

## 4. Item Popularity Distribution (Power Law)

A small percentage of items account for the vast majority of interactions — a classic
power-law (Pareto) distribution. The top 10% of items likely receive 70-80% of all reviews.
This motivates **popularity-based fallback** for cold-start users.

In [ ]:
item_counts = interactions.groupby('item_id').size().sort_values(ascending=False)
item_counts_reset = item_counts.reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Linear scale
axes[0].plot(item_counts_reset.values, color='steelblue', linewidth=0.8)
axes[0].set_xlabel('Item rank')
axes[0].set_ylabel('Review count')
axes[0].set_title('Item Popularity (linear)')

# Log-log scale to confirm power law
axes[1].loglog(item_counts_reset.index + 1, item_counts_reset.values, color='tomato', linewidth=0.8)
axes[1].set_xlabel('Item rank (log)')
axes[1].set_ylabel('Review count (log)')
axes[1].set_title('Item Popularity (log-log → confirms power law)')

plt.tight_layout()
plt.show()

top10_pct = int(len(item_counts) * 0.1)
top10_share = item_counts.iloc[:top10_pct].sum() / item_counts.sum()
print(f'Top 10% of items account for {top10_share:.1%} of all interactions')

## 5. Temporal Patterns

Are there seasonal spikes or weekday patterns? Understanding temporal structure helps
decide if time-based features (e.g., day-of-week, hour) should be included in the ranker.

In [ ]:
df_time = interactions.copy()
# Timestamps may be in milliseconds or seconds — normalise to seconds
if df_time['timestamp'].median() > 1e12:
    df_time['timestamp'] = df_time['timestamp'] / 1000
df_time['date'] = pd.to_datetime(df_time['timestamp'], unit='s', errors='coerce')
df_time = df_time.dropna(subset=['date'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Monthly interactions over time
monthly = df_time.set_index('date').resample('ME').size()
monthly.plot(ax=axes[0], color='steelblue')
axes[0].set_title('Monthly Interactions Over Time')
axes[0].set_ylabel('Interaction count')
axes[0].set_xlabel('')

# Day-of-week pattern
dow = df_time['date'].dt.day_name()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_counts = dow.value_counts().reindex(day_order)
dow_counts.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Interactions by Day of Week')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Summary of Findings

| Observation | Modeling implication |
|---|---|
| Sparsity > 99% | Use ALS implicit feedback CF, not SVD on ratings |
| J-shaped rating distribution | Treat ratings as confidence weights, not labels |
| Long-tail user activity | TTL-based session store is the right abstraction |
| Power-law item popularity | Popularity fallback is a strong cold-start baseline |
| Temporal spikes (holiday seasons) | Include `time_of_day` feature in XGBoost ranker |